# 🗄️ Mock Data Generator
> All fields generated via **LLM** (hg pipelines)  

**Steps:**
1. Run **Cell 1** to install dependencies
2. Run **Cell 2** to set up the runtime check
3. Run **Cell 3** to define all helpers & the LLM pipeline
4. Run **Cell 4** to launch the Gradio app

> 💡 For best results, set your Colab runtime to **T4 GPU** (*Runtime → Change runtime type → T4 GPU*) before starting.

## Cell 1 · Install dependencies

In [ ]:
%pip install -q gradio transformers accelerate bitsandbytes huggingface_hub python-dotenv
print('Dependencies installed')

## Cell 2 · Runtime check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU detected: {gpu}  ({mem:.1f} GB VRAM)')
    print('Recommended model: Qwen2.5-7B-Instruct or smaller on T4')
else:
    print(' No GPU found — using CPU. Generation will be very slow.')
    print('   Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    hf_token = os.getenv('HF_TOKEN')

print(f"Secret loaded: {hf_token is not None}")


In [ ]:
from huggingface_hub import login
login(hf_token, add_to_git_credential=True)

In [ ]:
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

## Cell 3 · Define pipeline & generation logic

In [ ]:
import gradio as gr
import json, csv, io
import torch

_pipeline = None
_loaded_model_id = None

def get_pipeline(model_id: str):
    """Load (or return cached) text-generation pipeline. Downloads on first call."""
    global _pipeline, _loaded_model_id
    if _pipeline is None or _loaded_model_id != model_id:
        print(f'Loading {model_id} — this may take a few minutes on first run...')
        from transformers import pipeline
        _pipeline = pipeline(
            "text-generation",
            model=model_id,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        _loaded_model_id = model_id
        print(f' {model_id} loaded')
    return _pipeline


In [ ]:

def parse_schema_from_description(description: str, model_id: str) -> dict:
    """
    Ask the LLM to interpret a natural-language schema description and return
    a dict with key 'fields' — a list of {name, description} objects for every
    field the user wants generated.
    """

    system = (
        "You are a data schema parser. Given a natural-language description of mock data, "
        "you identify every field the user wants and return ONLY a JSON object — no explanation, "
        "use snake_case for field names."
        "no markdown fences.\n"
        "Return format: {\"fields\": [{\"name\": \"<field_name>\", \"description\": \"<what it should contain>\"}]}\n"
        "Include ALL fields the user mentions or implies. Use concise, lowercase field names."
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": description},
    ]
    pipe_obj = get_pipeline(model_id)
    out = pipe_obj(
        messages,
        max_new_tokens=2000,
        temperature=0.2,
        do_sample=False,
        pad_token_id=pipe_obj.tokenizer.eos_token_id,
    )
    raw = out[0]["generated_text"][-1]["content"].strip()
    print(raw)
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())


In [ ]:

def llm_generate_records(fields: list, num_records: int, model_id: str) -> list:
    """Generate num_records rows for ALL fields using the LLM."""
    pipe = get_pipeline(model_id)

    field_specs = []
    for f in fields:
        field_specs.append(f'"{f["name"]}": {f["description"]}')

    keys_json = ", ".join(f'"{f["name"]}"' for f in fields)

    prompt = (
        f"Generate {num_records} realistic and diverse mock data records.\n"
        f"Each record must be a JSON object with exactly these keys: {keys_json}.\n"
        "Field rules:\n" +
        "\n".join(f"- {s}" for s in field_specs) + "\n"
        f"Return ONLY a JSON array of {num_records} objects, no explanation, no markdown fences."
    )

    messages = [{"role": "user", "content": prompt}]
    out = pipe(
        messages,
        max_new_tokens=min(4096, num_records * 250),
        temperature=0.9,
        do_sample=True,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )
    raw = out[0]["generated_text"][-1]["content"].strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    records = json.loads(raw.strip())
    return records[:num_records]

## Cell 4 · Launch Gradio app

In [ ]:
PRESETS = [
    ("User profiles","Generate user profiles with full name, email, phone number, and date of birth"),
    ("Company directory","Company directory with company name, address, employee name, salary, and active status"),
    ("E-commerce orders","E-commerce order records with customer name, email, order ID, order amount, and shipped status"),
    ("HR dataset","HR dataset: employee name, company, email, date of birth, salary, employee ID, and phone"),
    ("Global contacts","International contacts list with full name from diverse cultures, email, phone, and mailing address"),
    ("Patient records","Medical patient records with full name, date of birth, patient ID, phone number, email, and current patient status"),
    ("Student roster","University student roster with student name, email, student ID, date of birth, GPA (0.0–4.0), and enrolled status"),
    ("Financial accounts","Bank account records with account holder name, email, account ID, account balance, phone, and account status"),
    ("Logistics / shipping","Shipping records with recipient name, delivery address, tracking ID, phone, courier name, and delivered status"),
    ("Developer team","Engineering team directory with developer name, work email, team name, GitHub-style unique ID, salary, and employment status"),
    ("Hotel guests","Hotel guest records with guest name, email, phone, home address, booking ID, and checked-in status"),
    ("Product catalog","Product catalog with product name, SKU, price, in-stock status, and supplier address"),
    ("Game player profiles","Online game player profiles with username, email, player ID, account creation date, score, and online status"),
    ("Newsletter subscribers","Email newsletter list with subscriber name, email, phone, subscription date, and subscription status"),
    ("Gym members","Fitness club membership records with member name, email, phone, date of birth, membership ID, monthly fee, and membership status"),
]
PRESET_LABELS = [label for label, _ in PRESETS]
PRESET_MAP    = {label: prompt for label, prompt in PRESETS}


In [ ]:

with gr.Blocks(
    title="Mock Data Generator",
    theme=gr.themes.Soft(primary_hue="violet"),
    css="""
    .header { text-align:center; padding: 1rem 0 0.5rem; }
    .header h1 { font-size: 1.8rem; font-weight: 700; margin-bottom: 0.2rem; }
    .header p  { color: #666; font-size: 0.95rem; }
    #preview textarea { font-family: monospace; font-size: 0.78rem; }
    .preset-btn { font-size: 0.78rem !important; }
""",
) as demo:

    gr.HTML("""
        <div class="header">
            <h1>Mock Data Generator</h1>
            <p>Describe your data in plain English · all fields generated by the LLM</p>
        </div>
""")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Configuration")

            description_box = gr.Textbox(
                label=None,
                placeholder='e.g. "User profiles with name, email, phone, and salary"',
                lines=4,
                max_lines=8,
            )

            preset_dropdown = gr.Dropdown(
                label="Quick presets",
                choices=PRESET_LABELS,
                value=None,
                allow_custom_value=False,
                scale=1,
            )

            gr.HTML("""
                <div style="background:#fefce8;border:1px solid #fcd34d;border-radius:10px;padding:10px 14px 8px;margin-top:10px;margin-bottom:8px;">
                  <div style="font-size:0.78rem;font-weight:700;color:#92400e;margin-bottom:4px;">Model</div>
                  <div style="font-size:0.71rem;color:#78350f;line-height:1.5;">
                    Downloaded &amp; cached on first run. Use <strong>T4 GPU</strong> runtime for best speed.
                  </div>
                </div>
""")
            model_box = gr.Dropdown(
                label=None,
                choices=[
                   LLAMA,
                   QWEN,
                   GEMMA,
                   DEEPSEEK, 
                ],
                value=GEMMA,
                allow_custom_value=False,
            )

            gr.Markdown("---")
            output_format = gr.Radio(label="Output Format", choices=["JSON","CSV","SQL","Python"], value="JSON")
            generate_btn = gr.Button("Generate", variant="primary", size="lg")

        with gr.Column(scale=2):
            gr.Markdown("### Output")
            num_records = gr.Slider(label="Number of Records", minimum=1, maximum=100, value=5, step=1)
            stats_md    = gr.Markdown("_Stats will appear here after generation._")
            preview_box = gr.Code(label="Preview", language="json", lines=22, elem_id="preview", interactive=False)
            with gr.Row():
                copy_btn = gr.Button("Copy to clipboard", size="sm")
                dl_file  = gr.File(label="Download", visible=False)
            full_text = gr.State("")

    def on_preset_select(label):
        return PRESET_MAP.get(label, "")

    preset_dropdown.change(fn=on_preset_select, inputs=preset_dropdown, outputs=description_box)

    def on_generate(num, desc, fmt, model_id):
        import tempfile
        preview, text, stats = generate_data(num, desc, fmt, model_id)
        if not text:
            return preview, "", stats, gr.update(visible=False)
        ext = {"JSON":"json","CSV":"csv","SQL":"sql","Python":"py"}.get(fmt,"txt")
        tmp_path = os.path.join(tempfile.gettempdir(), f"mock_data.{ext}")
        with open(tmp_path, "w") as f:
            f.write(text)
        return preview, text, stats, gr.update(value=tmp_path, visible=True)

    generate_btn.click(
        on_generate,
        inputs=[num_records, description_box, output_format, model_box],
        outputs=[preview_box, full_text, stats_md, dl_file],
    )

    def update_lang(fmt):
        lang_map = {"JSON":"json","CSV":"markdown","SQL":"sql","Python":"python"}
        return gr.update(language=lang_map.get(fmt, "markdown"))

    output_format.change(update_lang, inputs=output_format, outputs=preview_box)
    copy_btn.click(None, inputs=full_text, js="(t) => { navigator.clipboard.writeText(t); return t; }")

demo.launch(share=True, debug=True)
